# Document Preparation Pipeline: Bronze -> Silver -> Gold
Cleans and chunks text documents into an embedding-ready format for downstream RAG / LLM applications.

**Architecture:** Raw documents -> Bronze (raw ingest) -> Silver (cleaned) -> Gold (chunked, embedding-ready)

## Step 1: Load sample dataset
Using a self-contained sample dataset (15 short tech articles) to avoid any external
network dependency during the pipeline build. Swap this for a Kaggle/HF dataset later if desired.

In [0]:
from pyspark.sql import Row

sample_articles = [
    {"doc_id": 1, "title": "Introduction to Cloud Computing", "text": "Cloud computing delivers computing services over the internet, including servers, storage, databases, networking, software, and analytics. Instead of owning physical infrastructure, organizations rent access to these resources from a cloud provider. This model offers flexibility, since businesses can scale resources up or down based on demand. The three primary service models are Infrastructure as a Service, Platform as a Service, and Software as a Service. Public clouds like Azure, AWS, and Google Cloud have made it possible for even small startups to access enterprise-grade infrastructure without major upfront investment."},
    {"doc_id": 2, "title": "What is Data Engineering", "text": "Data engineering is the practice of designing and building systems that collect, store, and analyze data at scale. Data engineers build pipelines that transform raw data into a usable format for analysts and data scientists. Core responsibilities include building ETL and ELT pipelines, managing data warehouses and lakes, and ensuring data quality and reliability. Modern data engineering increasingly relies on tools like Apache Spark, Delta Lake, and orchestration frameworks such as Airflow to handle growing data volumes efficiently."},
    {"doc_id": 3, "title": "Understanding Large Language Models", "text": "Large language models, or LLMs, are AI systems trained on massive amounts of text data to understand and generate human-like language. They work by predicting the next word in a sequence based on patterns learned during training. Modern LLMs like Claude and GPT can perform a wide range of tasks including summarization, translation, question answering, and code generation. These models are typically built using transformer architectures, which allow them to process relationships between words regardless of their distance in a sentence."},
    {"doc_id": 4, "title": "The Basics of Delta Lake", "text": "Delta Lake is an open-source storage layer that brings reliability to data lakes. It adds ACID transactions, scalable metadata handling, and unifies streaming and batch data processing on top of existing cloud storage. One of its most useful features is time travel, which allows users to query previous versions of a table. Delta Lake also supports schema enforcement and evolution, helping teams avoid data quality issues as pipelines evolve over time. It has become a standard choice for building medallion architecture pipelines."},
    {"doc_id": 5, "title": "What is Retrieval-Augmented Generation", "text": "Retrieval-Augmented Generation, or RAG, is a technique that combines a retrieval system with a language model to produce more accurate and grounded responses. Instead of relying solely on what the model learned during training, RAG retrieves relevant documents or chunks from an external knowledge base and provides them as context to the model before generating an answer. This approach helps reduce hallucinations and allows models to answer questions about information they were never trained on, such as private company documents."},
    {"doc_id": 6, "title": "Introduction to Vector Embeddings", "text": "Vector embeddings are numerical representations of text, images, or other data that capture semantic meaning in a high-dimensional space. Similar pieces of content end up close together in this space, which allows systems to find related information through similarity search rather than exact keyword matching. Embeddings are a core building block of modern search and recommendation systems, and they are essential for retrieval-augmented generation pipelines, where relevant text chunks need to be found quickly from a large knowledge base."},
    {"doc_id": 7, "title": "Azure AI Document Intelligence Overview", "text": "Azure AI Document Intelligence is a cloud service that extracts structured data from documents such as invoices, receipts, and forms. It uses prebuilt and custom machine learning models to identify fields like vendor names, dates, totals, and line items from scanned or digital documents. This removes the need for manual data entry and allows businesses to automate document-heavy workflows such as accounts payable processing, claims handling, and compliance reviews."},
    {"doc_id": 8, "title": "The Medallion Architecture Pattern", "text": "The medallion architecture is a data design pattern used to organize data in a lakehouse into three layers: bronze, silver, and gold. The bronze layer stores raw, unprocessed data exactly as it was ingested. The silver layer contains cleaned and validated data with duplicates removed and types standardized. The gold layer holds business-ready, aggregated data optimized for reporting or downstream applications. This layered approach makes pipelines easier to debug, since issues can be traced back to the specific stage where they were introduced."},
    {"doc_id": 9, "title": "Why Data Quality Checks Matter", "text": "Data quality checks are automated validations that catch problems in a dataset before they reach downstream users or systems. Common checks include verifying that required fields are not null, confirming row counts fall within expected ranges, and checking that values fall within valid bounds. Without these checks, silent data quality issues can lead to incorrect business decisions, broken dashboards, or unreliable machine learning models. Building quality checks directly into a pipeline, rather than relying on manual review, is considered a best practice in modern data engineering."},
    {"doc_id": 10, "title": "Chunking Strategies for RAG Systems", "text": "Chunking is the process of splitting long documents into smaller pieces before generating embeddings for retrieval. The size of each chunk matters: chunks that are too large can dilute relevance, while chunks that are too small may lose important context. A common strategy is to use a fixed word or token count per chunk, with some overlap between consecutive chunks to preserve context across boundaries. The right chunking strategy often depends on the type of content, such as whether it is narrative text, structured documents, or code."},
    {"doc_id": 11, "title": "Claude and Anthropic Overview", "text": "Claude is a family of large language models developed by Anthropic, designed with a strong focus on safety and helpfulness. Claude models can be accessed through an API, as well as through consumer products like Claude.ai, and through developer tools such as Claude Code. Anthropic designs its models to be helpful, honest, and harmless, incorporating extensive safety testing before release. Claude is commonly used for tasks including document analysis, coding assistance, customer support automation, and content generation."},
    {"doc_id": 12, "title": "Batch vs Streaming Data Processing", "text": "Batch processing handles data in large groups at scheduled intervals, such as once an hour or once a day, making it well suited for reporting and historical analysis. Streaming processing, by contrast, handles data continuously as it arrives, enabling near real-time insights for use cases like fraud detection or live dashboards. Many modern data platforms, including Apache Spark, support both batch and streaming workloads using a unified programming model, allowing teams to choose the right approach for each use case without maintaining two separate systems."},
    {"doc_id": 13, "title": "Understanding Data Lakehouses", "text": "A data lakehouse combines the low-cost, flexible storage of a data lake with the data management and transactional features typically found in a data warehouse. This architecture allows organizations to store structured, semi-structured, and unstructured data together while still supporting reliable SQL analytics and machine learning workloads. Technologies like Delta Lake and Apache Iceberg have made the lakehouse pattern practical at scale, reducing the need to maintain separate systems for analytics and raw data storage."},
    {"doc_id": 14, "title": "Common ETL Pipeline Failures", "text": "ETL pipelines can fail for many reasons, including schema changes in source systems, unexpected null values, duplicate records, and network timeouts during data transfer. A robust pipeline anticipates these failures by including validation steps, retry logic, and clear logging at each stage. Without proper monitoring, pipeline failures can go unnoticed for days, leading to stale or incomplete data reaching business users. Building observability into a pipeline from the start is far easier than adding it after problems arise in production."},
    {"doc_id": 15, "title": "Introduction to Prompt Engineering", "text": "Prompt engineering is the practice of crafting inputs to a language model in order to get more accurate, relevant, or structured outputs. Effective prompts often include clear instructions, relevant context, and examples of the desired output format. Techniques such as chain-of-thought prompting, where a model is asked to reason step by step, can improve performance on complex tasks. As language models are increasingly embedded into production applications, prompt engineering has become an important skill for developers building AI-powered features."},
]

bronze_df = spark.createDataFrame([Row(**a) for a in sample_articles])
display(bronze_df)
print(f"Bronze row count: {bronze_df.count()}")

doc_id,title,text
1,Introduction to Cloud Computing,"Cloud computing delivers computing services over the internet, including servers, storage, databases, networking, software, and analytics. Instead of owning physical infrastructure, organizations rent access to these resources from a cloud provider. This model offers flexibility, since businesses can scale resources up or down based on demand. The three primary service models are Infrastructure as a Service, Platform as a Service, and Software as a Service. Public clouds like Azure, AWS, and Google Cloud have made it possible for even small startups to access enterprise-grade infrastructure without major upfront investment."
2,What is Data Engineering,"Data engineering is the practice of designing and building systems that collect, store, and analyze data at scale. Data engineers build pipelines that transform raw data into a usable format for analysts and data scientists. Core responsibilities include building ETL and ELT pipelines, managing data warehouses and lakes, and ensuring data quality and reliability. Modern data engineering increasingly relies on tools like Apache Spark, Delta Lake, and orchestration frameworks such as Airflow to handle growing data volumes efficiently."
3,Understanding Large Language Models,"Large language models, or LLMs, are AI systems trained on massive amounts of text data to understand and generate human-like language. They work by predicting the next word in a sequence based on patterns learned during training. Modern LLMs like Claude and GPT can perform a wide range of tasks including summarization, translation, question answering, and code generation. These models are typically built using transformer architectures, which allow them to process relationships between words regardless of their distance in a sentence."
4,The Basics of Delta Lake,"Delta Lake is an open-source storage layer that brings reliability to data lakes. It adds ACID transactions, scalable metadata handling, and unifies streaming and batch data processing on top of existing cloud storage. One of its most useful features is time travel, which allows users to query previous versions of a table. Delta Lake also supports schema enforcement and evolution, helping teams avoid data quality issues as pipelines evolve over time. It has become a standard choice for building medallion architecture pipelines."
5,What is Retrieval-Augmented Generation,"Retrieval-Augmented Generation, or RAG, is a technique that combines a retrieval system with a language model to produce more accurate and grounded responses. Instead of relying solely on what the model learned during training, RAG retrieves relevant documents or chunks from an external knowledge base and provides them as context to the model before generating an answer. This approach helps reduce hallucinations and allows models to answer questions about information they were never trained on, such as private company documents."
6,Introduction to Vector Embeddings,"Vector embeddings are numerical representations of text, images, or other data that capture semantic meaning in a high-dimensional space. Similar pieces of content end up close together in this space, which allows systems to find related information through similarity search rather than exact keyword matching. Embeddings are a core building block of modern search and recommendation systems, and they are essential for retrieval-augmented generation pipelines, where relevant text chunks need to be found quickly from a large knowledge base."
7,Azure AI Document Intelligence Overview,"Azure AI Document Intelligence is a cloud service that extracts structured data from documents such as invoices, receipts, and forms. It uses prebuilt and custom machine learning models to identify fields like vendor names, dates, totals, and line items from scanned or digital documents. This removes the need for manual data entry and allows businesses to automate document-heavy workflows such as

Bronze row count: 15


## Step 2: Write Bronze layer (raw, untouched)

In [0]:
bronze_df.write.format("delta").mode("overwrite").saveAsTable("bronze_documents")
print("Bronze table written.")
display(spark.sql("SELECT * FROM bronze_documents LIMIT 5"))

Bronze table written.


doc_id,title,text
1,Introduction to Cloud Computing,"Cloud computing delivers computing services over the internet, including servers, storage, databases, networking, software, and analytics. Instead of owning physical infrastructure, organizations rent access to these resources from a cloud provider. This model offers flexibility, since businesses can scale resources up or down based on demand. The three primary service models are Infrastructure as a Service, Platform as a Service, and Software as a Service. Public clouds like Azure, AWS, and Google Cloud have made it possible for even small startups to access enterprise-grade infrastructure without major upfront investment."
2,What is Data Engineering,"Data engineering is the practice of designing and building systems that collect, store, and analyze data at scale. Data engineers build pipelines that transform raw data into a usable format for analysts and data scientists. Core responsibilities include building ETL and ELT pipelines, managing data warehouses and lakes, and ensuring data quality and reliability. Modern data engineering increasingly relies on tools like Apache Spark, Delta Lake, and orchestration frameworks such as Airflow to handle growing data volumes efficiently."
3,Understanding Large Language Models,"Large language models, or LLMs, are AI systems trained on massive amounts of text data to understand and generate human-like language. They work by predicting the next word in a sequence based on patterns learned during training. Modern LLMs like Claude and GPT can perform a wide range of tasks including summarization, translation, question answering, and code generation. These models are typically built using transformer architectures, which allow them to process relationships between words regardless of their distance in a sentence."
4,The Basics of Delta Lake,"Delta Lake is an open-source storage layer that brings reliability to data lakes. It adds ACID transactions, scalable metadata handling, and unifies streaming and batch data processing on top of existing cloud storage. One of its most useful features is time travel, which allows users to query previous versions of a table. Delta Lake also supports schema enforcement and evolution, helping teams avoid data quality issues as pipelines evolve over time. It has become a standard choice for building medallion architecture pipelines."
5,What is Retrieval-Augmented Generation,"Retrieval-Augmented Generation, or RAG, is a technique that combines a retrieval system with a language model to produce more accurate and grounded responses. Instead of relying solely on what the model learned during training, RAG retrieves relevant documents or chunks from an external knowledge base and provides them as context to the model before generating an answer. This approach helps reduce hallucinations and allows models to answer questions about information they were never trained on, such as private company documents."


## Step 3: Build Silver layer — clean and validate
Deduplicate, trim whitespace, drop rows with missing/short text, standardize columns.

In [0]:
from pyspark.sql.functions import col, trim, length, lower, regexp_replace

silver_df = (
    spark.table("bronze_documents")
    .dropDuplicates(["doc_id"])
    .withColumn("title", trim(col("title")))
    .withColumn("text", trim(regexp_replace(col("text"), r"\s+", " ")))
    .filter(col("text").isNotNull())
    .filter(length(col("text")) > 20)
)

silver_df.write.format("delta").mode("overwrite").saveAsTable("silver_documents")

# Data quality checks
silver_count = silver_df.count()
null_text_count = silver_df.filter(col("text").isNull()).count()

assert silver_count > 0, "Silver layer is empty — pipeline failed."
assert null_text_count == 0, "Found null text values in Silver layer."

print(f"Silver row count: {silver_count}")
print(f"Null text rows: {null_text_count}")
print("Data quality checks passed.")
display(spark.sql("SELECT * FROM silver_documents LIMIT 5"))

Silver row count: 15
Null text rows: 0
Data quality checks passed.


doc_id,title,text
9,Why Data Quality Checks Matter,"Data quality checks are automated validations that catch problems in a dataset before they reach downstream users or systems. Common checks include verifying that required fields are not null, confirming row counts fall within expected ranges, and checking that values fall within valid bounds. Without these checks, silent data quality issues can lead to incorrect business decisions, broken dashboards, or unreliable machine learning models. Building quality checks directly into a pipeline, rather than relying on manual review, is considered a best practice in modern data engineering."
11,Claude and Anthropic Overview,"Claude is a family of large language models developed by Anthropic, designed with a strong focus on safety and helpfulness. Claude models can be accessed through an API, as well as through consumer products like Claude.ai, and through developer tools such as Claude Code. Anthropic designs its models to be helpful, honest, and harmless, incorporating extensive safety testing before release. Claude is commonly used for tasks including document analysis, coding assistance, customer support automation, and content generation."
14,Common ETL Pipeline Failures,"ETL pipelines can fail for many reasons, including schema changes in source systems, unexpected null values, duplicate records, and network timeouts during data transfer. A robust pipeline anticipates these failures by including validation steps, retry logic, and clear logging at each stage. Without proper monitoring, pipeline failures can go unnoticed for days, leading to stale or incomplete data reaching business users. Building observability into a pipeline from the start is far easier than adding it after problems arise in production."
3,Understanding Large Language Models,"Large language models, or LLMs, are AI systems trained on massive amounts of text data to understand and generate human-like language. They work by predicting the next word in a sequence based on patterns learned during training. Modern LLMs like Claude and GPT can perform a wide range of tasks including summarization, translation, question answering, and code generation. These models are typically built using transformer architectures, which allow them to process relationships between words regardless of their distance in a sentence."
4,The Basics of Delta Lake,"Delta Lake is an open-source storage layer that brings reliability to data lakes. It adds ACID transactions, scalable metadata handling, and unifies streaming and batch data processing on top of existing cloud storage. One of its most useful features is time travel, which allows users to query previous versions of a table. Delta Lake also supports schema enforcement and evolution, helping teams avoid data quality issues as pipelines evolve over time. It has become a standard choice for building medallion architecture pipelines."


## Step 4: Build Gold layer — chunk text for embedding
Splits each document into ~60-word chunks with a small overlap, producing an
embedding-ready table that could feed a RAG pipeline (e.g. the Claude-based project).

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
import pandas as pd

CHUNK_SIZE_WORDS = 60
OVERLAP_WORDS = 15

def chunk_text(doc_id: int, title: str, text: str):
    words = text.split()
    chunks = []
    start = 0
    idx = 0
    while start < len(words):
        end = min(start + CHUNK_SIZE_WORDS, len(words))
        chunk_words = words[start:end]
        chunks.append({
            "doc_id": doc_id,
            "title": title,
            "chunk_index": idx,
            "chunk_text": " ".join(chunk_words),
            "word_count": len(chunk_words),
        })
        idx += 1
        if end == len(words):
            break
        start += CHUNK_SIZE_WORDS - OVERLAP_WORDS
    return chunks

silver_pd = silver_df.select("doc_id", "title", "text").toPandas()

all_chunks = []
for _, row in silver_pd.iterrows():
    all_chunks.extend(chunk_text(row["doc_id"], row["title"], row["text"]))

chunks_pd = pd.DataFrame(all_chunks)
chunks_pd["chunk_id"] = chunks_pd["doc_id"].astype(str) + "_" + chunks_pd["chunk_index"].astype(str)
chunks_pd = chunks_pd[["chunk_id", "doc_id", "title", "chunk_index", "chunk_text", "word_count"]]

gold_df = spark.createDataFrame(chunks_pd)
gold_df.write.format("delta").mode("overwrite").saveAsTable("gold_chunks")

print(f"Gold row count (chunks): {gold_df.count()}")
display(spark.sql("SELECT * FROM gold_chunks LIMIT 10"))

Gold row count (chunks): 30


chunk_id,doc_id,title,chunk_index,chunk_text,word_count
1_0,1,Introduction to Cloud Computing,0,"Cloud computing delivers computing services over the internet, including servers, storage, databases, networking, software, and analytics. Instead of owning physical infrastructure, organizations rent access to these resources from a cloud provider. This model offers flexibility, since businesses can scale resources up or down based on demand. The three primary service models are Infrastructure as a Service, Platform as a Service,",60
1_1,1,Introduction to Cloud Computing,1,"demand. The three primary service models are Infrastructure as a Service, Platform as a Service, and Software as a Service. Public clouds like Azure, AWS, and Google Cloud have made it possible for even small startups to access enterprise-grade infrastructure without major upfront investment.",44
2_0,2,What is Data Engineering,0,"Data engineering is the practice of designing and building systems that collect, store, and analyze data at scale. Data engineers build pipelines that transform raw data into a usable format for analysts and data scientists. Core responsibilities include building ETL and ELT pipelines, managing data warehouses and lakes, and ensuring data quality and reliability. Modern data engineering increasingly relies on",60
2_1,2,What is Data Engineering,1,"warehouses and lakes, and ensuring data quality and reliability. Modern data engineering increasingly relies on tools like Apache Spark, Delta Lake, and orchestration frameworks such as Airflow to handle growing data volumes efficiently.",33
3_0,3,Understanding Large Language Models,0,"Large language models, or LLMs, are AI systems trained on massive amounts of text data to understand and generate human-like language. They work by predicting the next word in a sequence based on patterns learned during training. Modern LLMs like Claude and GPT can perform a wide range of tasks including summarization, translation, question answering, and code generation. These models",60
3_1,3,Understanding Large Language Models,1,"a wide range of tasks including summarization, translation, question answering, and code generation. These models are typically built using transformer architectures, which allow them to process relationships between words regardless of their distance in a sentence.",36
4_0,4,The Basics of Delta Lake,0,"Delta Lake is an open-source storage layer that brings reliability to data lakes. It adds ACID transactions, scalable metadata handling, and unifies streaming and batch data processing on top of existing cloud storage. One of its most useful features is time travel, which allows users to query previous versions of a table. Delta Lake also supports schema enforcement and evolution,",60
4_1,4,The Basics of Delta Lake,1,"to query previous versions of a table. Delta Lake also supports schema enforcement and evolution, helping teams avoid data quality issues as pipelines evolve over time. It has become a standard choice for building medallion architecture pipelines.",37
5_0,5,What is Retrieval-Augmented Generation,0,"Retrieval-Augmented Generation, or RAG, is a technique that combines a retrieval system with a language model to produce more accurate and grounded responses. Instead of relying solely on what the model learned during training, RAG retrieves relevant documents or chunks from an external knowledge base and provides them as context to the model before generating an answer. This approach helps",60
5_1,5,What is Retrieval-Augmented Generation,1,"and provides them as context to the model before generating an answer. This approach helps reduce hallucinations and allows models to answer questions about information they were never trained on, such as private company documents.",35


## Step 5: Validation summary
Row counts across each layer, plus a sample of the final embedding-ready chunks.

In [0]:
bronze_count = spark.table("bronze_documents").count()
silver_count = spark.table("silver_documents").count()
gold_count = spark.table("gold_chunks").count()
avg_chunk_len = spark.sql("SELECT AVG(word_count) as avg_words FROM gold_chunks").collect()[0]["avg_words"]

print("Pipeline Summary")
print("-" * 40)
print(f"Bronze (raw documents):     {bronze_count}")
print(f"Silver (cleaned documents): {silver_count}")
print(f"Gold (embedding chunks):    {gold_count}")
print(f"Average chunk length:       {avg_chunk_len:.1f} words")
print("-" * 40)
print("\nSample chunks ready for embedding:\n")

sample_chunks = spark.sql("SELECT chunk_id, title, chunk_text FROM gold_chunks LIMIT 3").collect()
for row in sample_chunks:
    print(f"[{row['chunk_id']}] ({row['title']})")
    print(f"  {row['chunk_text'][:150]}...")
    print()

Pipeline Summary
----------------------------------------
Bronze (raw documents):     15
Silver (cleaned documents): 15
Gold (embedding chunks):    30
Average chunk length:       47.8 words
----------------------------------------

Sample chunks ready for embedding:

[1_0] (Introduction to Cloud Computing)
  Cloud computing delivers computing services over the internet, including servers, storage, databases, networking, software, and analytics. Instead of ...

[1_1] (Introduction to Cloud Computing)
  demand. The three primary service models are Infrastructure as a Service, Platform as a Service, and Software as a Service. Public clouds like Azure, ...

[2_0] (What is Data Engineering)
  Data engineering is the practice of designing and building systems that collect, store, and analyze data at scale. Data engineers build pipelines that...

